In [ ]:
import math
import time
import numpy as np
from numba import cuda

# --- Kernels ---

@cuda.jit
def kernel_a(y):
    """Kernel A: Uniform Path - All threads execute identical arithmetic."""
    idx = cuda.grid(1)
    if idx < y.size:
        val = y[idx]
        for _ in range(1000):
            val = val * 1.0001 + 0.0001
        y[idx] = val

@cuda.jit
def kernel_b(y):
    """Kernel B: Full Divergence - Adjacent threads in warps diverge."""
    idx = cuda.grid(1)
    if idx < y.size:
        val = y[idx]
        if idx % 2 == 0:
            for _ in range(1000):
                val = val * 1.0001 + 0.0001
        else:
            for _ in range(1000):
                val = (val - 0.0001) / 1.0001
        y[idx] = val

@cuda.jit
def kernel_c(y):
    """Kernel C: Warp-Aligned Branching - Entire warps branch uniformly."""
    idx = cuda.grid(1)
    if idx < y.size:
        val = y[idx]
        warp_id = idx // 32
        if warp_id % 2 == 0:
            for _ in range(1000):
                val = val * 1.0001 + 0.0001
        else:
            for _ in range(1000):
                val = (val - 0.0001) / 1.0001
        y[idx] = val

# --- Benchmarking Utility ---

def benchmark_kernel(kernel, d_array, threads_per_block, blocks_per_grid):
    # Warm-up launch
    kernel[blocks_per_grid, threads_per_block](d_array)
    cuda.synchronize()

    # 10 Consecutive timed trials
    timings = []
    for _ in range(10):
        start = time.perf_counter()
        kernel[blocks_per_grid, threads_per_block](d_array)
        cuda.synchronize()
        end = time.perf_counter()
        timings.append((end - start) * 1000.0)  # Convert to ms

    return np.mean(timings)

def main():
    N = 2**20  # 1,048,576 elements
    h_data = np.ones(N, dtype=np.float32)

    threads_per_block = 256
    blocks_per_grid = math.ceil(N / threads_per_block)

    # Allocate Device Memory
    d_data = cuda.to_device(h_data)

    time_a = benchmark_kernel(kernel_a, d_data, threads_per_block, blocks_per_grid)
    time_b = benchmark_kernel(kernel_b, d_data, threads_per_block, blocks_per_grid)
    time_c = benchmark_kernel(kernel_c, d_data, threads_per_block, blocks_per_grid)

    print(f"Kernel A (Uniform Path):            {time_a:.3f} ms")
    print(f"Kernel B (Full Divergence):         {time_b:.3f} ms")
    print(f"Kernel C (Warp-Aligned Branching):  {time_c:.3f} ms")

if __name__ == "__main__":
    main()

Kernel A (Uniform Path):            20.829 ms
Kernel B (Full Divergence):         94.566 ms
Kernel C (Warp-Aligned Branching):  46.432 ms


In [ ]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def stencil_1d(d_in, d_out, N):
    """1D 3-point smoothing stencil with halo boundary clamping."""
    idx = cuda.grid(1)

    if idx < N:
        # Halo boundary clamping conditions
        if idx == 0:
            left = d_in[0]
            right = d_in[1] if N > 1 else d_in[0]
        elif idx == N - 1:
            left = d_in[N - 2]
            right = d_in[N - 1]
        else:
            left = d_in[idx - 1]
            right = d_in[idx + 1]

        center = d_in[idx]
        d_out[idx] = 0.25 * left + 0.5 * center + 0.25 * right

def cpu_stencil(arr):
    """Exact CPU reference validator using NumPy pad."""
    padded = np.pad(arr, (1, 1), mode='edge')
    return 0.25 * padded[:-2] + 0.5 * padded[1:-1] + 0.25 * padded[2:]

def run_stencil(h_in):
    """Host wrapper function handling device transfers and kernel execution."""
    N = len(h_in)
    threads_per_block = 256
    blocks_per_grid = math.ceil(N / threads_per_block)

    d_in = cuda.to_device(h_in.astype(np.float32))
    d_out = cuda.device_array(N, dtype=np.float32)

    stencil_1d[blocks_per_grid, threads_per_block](d_in, d_out, N)
    cuda.synchronize()

    return d_out.copy_to_host()

if __name__ == "__main__":
    N = 100_007
    test_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)

    h_out_gpu = run_stencil(test_in)
    cpu_ref = cpu_stencil(test_in)

    delta = np.max(np.abs(h_out_gpu - cpu_ref))
    assert np.allclose(h_out_gpu, cpu_ref, atol=1e-4)
    print(f"TASK 2 PASSED: MAX DELTA = {delta:.6e}")

TASK 2 PASSED: MAX DELTA = 5.960464e-08


In [ ]:
import numpy as np
from numba import cuda

@cuda.jit
def grid_stride_scale_kernel(d_arr, factor, N):
    """Grid-stride loop scaling arbitrary vector sizes."""
    start = cuda.grid(1)
    stride = cuda.gridsize(1)

    for i in range(start, N, stride):
        d_arr[i] = d_arr[i] * factor

def run_grid_stride(h_arr, factor):
    """Host caller function configuring constrained hardware launches."""
    N = len(h_arr)
    d_arr = cuda.to_device(h_arr.astype(np.float32))

    # Hardware constraint: strictly 256 threads/block and 64 blocks
    threads_per_block = 256
    blocks_per_grid = 64

    grid_stride_scale_kernel[blocks_per_grid, threads_per_block](d_arr, factor, N)
    cuda.synchronize()

    return d_arr.copy_to_host()

if __name__ == "__main__":
    N = 16_777_216  # 2^24 elements
    factor = 4.25
    test_arr = np.ones(N, dtype=np.float32)

    res = run_grid_stride(test_arr, factor)

    assert np.allclose(res, factor)
    print(f"TASK 3 PASSED: All {N} elements successfully scaled by {factor}.")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


TASK 3 PASSED: All 16777216 elements successfully scaled by 4.25.


In [ ]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def sobel_x_kernel(d_in, d_out, rows, cols):
    """2D Spatial Convolution - Sobel Horizontal Filter."""
    col, row = cuda.grid(2)

    if row < rows and col < cols:
        # Check internal pixels vs. border pixels
        if 0 < row < rows - 1 and 0 < col < cols - 1:
            d_out[row, col] = (
                -1.0 * d_in[row - 1, col - 1] + 1.0 * d_in[row - 1, col + 1]
                - 2.0 * d_in[row, col - 1]     + 2.0 * d_in[row, col + 1]
                - 1.0 * d_in[row + 1, col - 1] + 1.0 * d_in[row + 1, col + 1]
            )
        else:
            d_out[row, col] = 0.0

def run_sobel(h_img):
    """Host wrapper function returning filtered matrix."""
    rows, cols = h_img.shape
    d_in = cuda.to_device(h_img.astype(np.float32))
    d_out = cuda.device_array((rows, cols), dtype=np.float32)

    threads_2d = (16, 16)
    blocks_per_grid_x = math.ceil(cols / threads_2d[0])
    blocks_per_grid_y = math.ceil(rows / threads_2d[1])
    grid_2d = (blocks_per_grid_x, blocks_per_grid_y)

    sobel_x_kernel[grid_2d, threads_2d](d_in, d_out, rows, cols)
    cuda.synchronize()

    return d_out.copy_to_host()

if __name__ == "__main__":
    test_img = np.ones((2048, 2048), dtype=np.float32)
    sobel_res = run_sobel(test_img)

    assert np.max(np.abs(sobel_res[1:-1, 1:-1])) < 1e-5
    assert np.all(sobel_res[0, :] == 0.0)
    assert np.all(sobel_res[:, 0] == 0.0)
    print("TASK 4 PASSED: 2D Sobel Horizontal Filter calculated successfully.")

TASK 4 PASSED: 2D Sobel Horizontal Filter calculated successfully.


# CUDA Lab 02: Advanced Geometries & Stencils

**Student ID:** 230109015
**Allocated GPU Node:** Tesla T4
**CUDA Compute Capability:** 7.5
**Official Verification Token:** <PASTE_TOKEN_HERE>

## Task 1: Warp Divergence Benchmark Results

| Kernel Variant | Execution Time (ms) | Description |
| :--- | :--- | :--- |
| **Kernel A (Uniform Path)** | 20.829 ms | Identical path across all threads in the warp |
| **Kernel B (Full Divergence)** | 94.566 ms | Interleaved threads diverge (`idx % 2 == 0`) |
| **Kernel C (Warp-Aligned)** | 46.432 ms | Entire warps execute unified paths (`warp_id % 2 == 0`) |

### Warp Divergence Analysis
- **Kernel A vs. Kernel B (~4.54x slowdown):** When intra-warp divergence occurs in Kernel B, the Streaming Multiprocessor (SM) cannot execute both branches simultaneously. It serializes the two execution paths—masking off active threads for each branch—causing thread execution throughput to drop significantly.
- **Kernel B vs. Kernel C (~2.04x speedup):** Kernel C eliminates intra-warp divergence by assigning branches based on `warp_id = idx // 32`. Because every 32-thread warp takes either Path 1 or Path 2 uniformly, execution paths within individual warps remain fully aligned without hardware path serialization.